# 05 — silver_products
**Entrées :** `bronze.products` + `bronze.product_category_name_translation` — **Sortie :** `silver.silver_products` (32 951 lignes)
**À faire :** renommer les colonnes mal orthographiées (`lenght` → `length`), mettre les nombres en entier,
remplacer la catégorie vide par une valeur claire, ajouter la traduction en `LEFT JOIN` (pour ne perdre aucun produit).
**Indicateurs :** `flag_category_missing` (610), `flag_translation_missing` (13), `flag_dimensions_missing` (2).

> Astuce : dans le CSV, les entiers sont écrits `40.0`. On passe donc par DOUBLE puis INT.

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
# Les tables Bronze sont supposées s'appeler : orders, customers, order_items, order_payments,
# order_reviews, products, sellers, product_category_name_translation.
# Si les tiennes ont d'autres noms, change-les dans les requêtes (Ctrl+F).

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    """Exécute une requête qui renvoie une seule valeur."""
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    """Compare ce qu'on obtient à ce qu'on attend (valeurs de ton document de travail)."""
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_products AS
WITH p AS (
    SELECT
        product_id,
        NULLIF(TRIM(product_category_name), '') AS category_pt,
        CAST(TRY_CAST(product_name_lenght        AS DOUBLE) AS INT) AS product_name_length,
        CAST(TRY_CAST(product_description_lenght AS DOUBLE) AS INT) AS product_description_length,
        CAST(TRY_CAST(product_photos_qty         AS DOUBLE) AS INT) AS product_photos_qty,
        CAST(TRY_CAST(product_weight_g           AS DOUBLE) AS INT) AS product_weight_g,
        CAST(TRY_CAST(product_length_cm          AS DOUBLE) AS INT) AS product_length_cm,
        CAST(TRY_CAST(product_height_cm          AS DOUBLE) AS INT) AS product_height_cm,
        CAST(TRY_CAST(product_width_cm           AS DOUBLE) AS INT) AS product_width_cm,
        _loaded_at
    FROM {BRONZE}.products
)
SELECT
    p.product_id,
    COALESCE(p.category_pt, 'categorie_inconnue') AS product_category_name,
    COALESCE(t.product_category_name_english,
             CASE WHEN p.category_pt IS NULL THEN 'unknown_category' END) AS product_category_name_english,
    p.product_name_length, p.product_description_length, p.product_photos_qty,
    p.product_weight_g, p.product_length_cm, p.product_height_cm, p.product_width_cm,
    p.category_pt IS NULL AS flag_category_missing,
    (p.category_pt IS NOT NULL AND t.product_category_name_english IS NULL) AS flag_translation_missing,
    (p.product_weight_g IS NULL OR p.product_length_cm IS NULL
     OR p.product_height_cm IS NULL OR p.product_width_cm IS NULL) AS flag_dimensions_missing,
    p._loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM p
LEFT JOIN {BRONZE}.category_translation t ON p.category_pt = t.product_category_name
""")

In [0]:
t = f"{SILVER}.silver_products"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 32951)
check("product_id doublons", scalar(f"SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM {t}"), 0)
check("flag_category_missing", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_category_missing"), 610)
check("flag_translation_missing", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_translation_missing"), 13)
check("flag_dimensions_missing", scalar(f"SELECT COUNT(*) FROM {t} WHERE flag_dimensions_missing"), 2)

## Vérifier que les 610 produits sans catégorie sont les mêmes que ceux sans nom ni photo
(Le document dit « probablement les mêmes lignes, à vérifier ».)

In [0]:
spark.sql(f"""
    SELECT COUNT(*) AS sans_categorie,
           SUM(CASE WHEN b.product_name_lenght IS NULL OR TRIM(b.product_name_lenght) = '' THEN 1 ELSE 0 END) AS aussi_sans_nom,
           SUM(CASE WHEN b.product_photos_qty  IS NULL OR TRIM(b.product_photos_qty)  = '' THEN 1 ELSE 0 END) AS aussi_sans_photos
    FROM {t} s JOIN {BRONZE}.products b ON s.product_id = b.product_id
    WHERE s.flag_category_missing
""").show()

## Aucun produit perdu par la jointure
Le test de comptage (32 951) le garantit : un `LEFT JOIN` sur une traduction unique ne change pas le nombre de lignes.

## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?